In [1]:
import sys
from pathlib import Path

# Garante que o pacote do projeto (src/spark_delta) seja encontrado mesmo que o
# kernel não tenha sido instalado com `uv sync` (fallback para src/).
try:
    import spark_delta  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
    sys.path.insert(0, str(Path.cwd().resolve() / "src"))

import io
import pandas as pd

from spark_delta.config import settings
from spark_delta.sqlserver import connect
from spark_delta.storage import s3_client, ensure_bucket, list_objects

print(f"SQL Server: {settings.db_server}:{settings.db_port}/{settings.db_database}")
print(f"MinIO:      {settings.minio_endpoint} | bucket: {settings.landing_bucket}")

SQL Server: localhost:1433/LojaDB
MinIO:      http://localhost:9020 | bucket: landing-zone


In [2]:
conn = connect(settings.db_database)
cursor = conn.cursor()

cursor.execute("""
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE' AND TABLE_SCHEMA = ?
    ORDER BY TABLE_NAME
""", settings.db_schema)
tabelas = [row[0] for row in cursor.fetchall()]

print(f"{len(tabelas)} tabelas encontradas:")
for t in tabelas:
    cursor.execute(f"SELECT COUNT(*) FROM [{settings.db_schema}].[{t}]")
    print(f"  - {t:<12} {cursor.fetchone()[0]:>6} registros")

7 tabelas encontradas:
  - categoria         8 registros
  - cliente        1000 registros
  - estoque         120 registros
  - fornecedor       25 registros
  - item_venda    10964 registros
  - produto         120 registros
  - venda          5000 registros


In [3]:
s3 = s3_client()
criado = ensure_bucket(s3, settings.landing_bucket)
print(f"Bucket [{settings.landing_bucket}]", "criado!" if criado else "já existia")
print("Buckets:", [b["Name"] for b in s3.list_buckets()["Buckets"]])

Bucket [landing-zone] criado!
Buckets: ['landing-zone']


In [4]:
resultados = []

for tabela in tabelas:
    cursor.execute(f"SELECT * FROM [{settings.db_schema}].[{tabela}]")
    colunas = [d[0] for d in cursor.description]
    df = pd.DataFrame.from_records(cursor.fetchall(), columns=colunas)

    buffer = io.StringIO()
    # formato fixo de data/hora: é o mesmo que o notebook 02 usa para ler
    df.to_csv(buffer, index=False, date_format="%Y-%m-%d %H:%M:%S")
    corpo = buffer.getvalue().encode("utf-8")

    chave = f"{tabela}.csv"
    s3.put_object(Bucket=settings.landing_bucket, Key=chave, Body=corpo, ContentType="text/csv")

    resultados.append({"tabela": tabela, "registros": len(df), "colunas": len(colunas),
                       "tamanho_kb": round(len(corpo) / 1024, 1)})
    print(f"  {tabela:<12} -> s3://{settings.landing_bucket}/{chave} ({len(df)} registros)")

print(f"\nExtração concluída: {len(tabelas)} tabelas.")

  categoria    -> s3://landing-zone/categoria.csv (8 registros)
  cliente      -> s3://landing-zone/cliente.csv (1000 registros)
  estoque      -> s3://landing-zone/estoque.csv (120 registros)
  fornecedor   -> s3://landing-zone/fornecedor.csv (25 registros)
  item_venda   -> s3://landing-zone/item_venda.csv (10964 registros)
  produto      -> s3://landing-zone/produto.csv (120 registros)
  venda        -> s3://landing-zone/venda.csv (5000 registros)

Extração concluída: 7 tabelas.


In [5]:
for obj in list_objects(s3, settings.landing_bucket):
    print(f"  {obj['Key']:<20} {obj['Size'] / 1024:>8.1f} KB")

resumo = pd.DataFrame(resultados)
print(f"\nTotal: {resumo['registros'].sum():,} registros | {resumo['tamanho_kb'].sum():,.1f} KB")
resumo

  categoria.csv             0.1 KB
  cliente.csv              89.6 KB
  estoque.csv               3.4 KB
  fornecedor.csv            1.4 KB
  item_venda.csv          186.4 KB
  produto.csv               5.8 KB
  venda.csv               228.5 KB

Total: 17,237 registros | 515.2 KB


,tabela,registros,colunas,tamanho_kb
0,categoria,8,2,0.1
1,cliente,1000,7,89.6
2,estoque,120,4,3.4
3,fornecedor,25,5,1.4
4,item_venda,10964,4,186.4
5,produto,120,6,5.8
6,venda,5000,5,228.5


In [6]:
# Conteúdo de um CSV direto do MinIO (primeiras linhas)
obj = s3.get_object(Bucket=settings.landing_bucket, Key="produto.csv")
print("\n".join(obj["Body"].read().decode("utf-8").splitlines()[:6]))

cd_produto,nm_produto,cd_categoria,cd_fornecedor,preco,dt_cadastro
1,Notebook 14 pol i5 Asus,1,9,8455.52,2024-03-02
2,Notebook 14 pol i5 Acer,1,4,5928.86,2022-01-29
3,Notebook 14 pol i5 HP,1,2,9211.51,2022-09-29
4,Notebook 15 pol i7 Acer,1,7,4766.00,2023-08-14
5,Notebook 15 pol i7 Asus,1,18,5803.39,2024-04-05


In [7]:
cursor.close()
conn.close()
print("Conexão com SQL Server encerrada.")

Conexão com SQL Server encerrada.
